In [1]:
!pip install -q \
  langchain \
  langchain-community \
  langchain-core \
  langchain-chroma \
  langchain-huggingface \
  sentence-transformers \
  chromadb \
  pyyaml \
  tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6

In [2]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [3]:
import shutil

shutil.rmtree("/content/drive/MyDrive/chroma_gitlab")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/chroma_gitlab'

In [4]:
import torch
print(torch.cuda.is_available())

True


In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
TARGET_DIRECTORIES = [
    "/content/drive/MyDrive/handbook/",
    "/content/drive/MyDrive/direction/"
]

In [9]:

PERSIST_DIR = "/content/drive/MyDrive/chroma_gitlab"

In [10]:
import re
import yaml
import shutil
from typing import List
from tqdm import tqdm

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [11]:
def clean_text(text: str) -> str:
    import re

    # Remove iframe completely
    text = re.sub(r"<iframe.*?>.*?</iframe>", "", text, flags=re.DOTALL)

    # Remove images
    text = re.sub(r"<img.*?>", "", text)
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Keep markdown text, remove links
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)

    # Remove comments
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)

    # Remove TOC
    text = re.sub(r"\[\[TOC\]\]", "", text)

    # Normalize whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def is_useful(text: str) -> bool:
    import re

    text = text.strip()

    # Too short → useless
    if len(text) < 80:
        return False

    # Count real words (ignore symbols)
    words = re.findall(r"\b[a-zA-Z]{3,}\b", text)

    if len(words) < 10:
        return False

    return True

In [12]:
def parse_frontmatter(content):
    if content.startswith("---"):
        parts = content.split("---", 2)
        if len(parts) >= 3:
            try:
                return yaml.safe_load(parts[1]) or {}, parts[2]
            except yaml.YAMLError:
                return {}, content
    return {}, content

In [13]:
def path_to_hierarchy(path: str) -> str:
    parts = path.split(os.sep)
    return " > ".join(parts[-4:])

In [14]:
def chunk_markdown(md_text: str):
    chunks = []
    stack = []
    current = []

    for line in md_text.split("\n"):
        if line.startswith("#"):
            if current:
                chunks.append((stack.copy(), "\n".join(current)))
                current = []

            level = len(line) - len(line.lstrip("#"))
            heading = line.strip("# ").strip()

            stack[:] = stack[:level-1]
            stack.append(heading)

        else:
            current.append(line)

    if current:
        chunks.append((stack.copy(), "\n".join(current)))

    return chunks

In [15]:
def split_large(text, max_len=700):
    paras = text.split("\n\n")
    out, curr = [], ""

    for p in paras:
        if len(curr) + len(p) < max_len:
            curr += "\n\n" + p
        else:
            if curr:
                out.append(curr.strip())
            curr = p

    if curr:
        out.append(curr.strip())

    return out

In [16]:
def process_file(path: str) -> List[Document]:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    fm, content = parse_frontmatter(raw)
    content = clean_text(content)

    title = fm.get("title") or os.path.basename(path)
    desc = fm.get("description", "")

    hierarchy = path_to_hierarchy(path)

    chunks = chunk_markdown(content)

    docs = []

    for headings, text in chunks:
        if not is_useful(text):
            continue

        section = " > ".join(headings) if headings else "General"

        sub_chunks = split_large(text)

        for sub in sub_chunks:
            formatted = f"""
Title: {title}
Description: {desc}

Path: {hierarchy}
Section: {section}

Content:
{sub}
"""

            docs.append(
                Document(
                    page_content=formatted.strip(),
                    metadata={
                        "source": path,
                        "section": section
                    }
                )
            )

    return docs

In [17]:
import os

In [18]:
from tqdm import tqdm

all_files = []

for d in TARGET_DIRECTORIES:
    for root, _, files in os.walk(d):
        for file in files:
            if file.endswith(".md"):
                all_files.append(os.path.join(root, file))

print("Total files:", len(all_files))

Total files: 4114


In [19]:
all_docs = []

for path in tqdm(all_files, desc="Processing markdown files"):
    try:
        docs = process_file(path)
        all_docs.extend(docs)
    except Exception as e:
        print(path, e)

print("Total chunks:", len(all_docs))

Processing markdown files: 100%|██████████| 4114/4114 [23:53<00:00,  2.87it/s]

Total chunks: 66580


In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
vectorstore = Chroma(
    collection_name="gitlab-final",
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR
)

In [22]:
BATCH_SIZE = 100

for i in tqdm(range(0, len(all_docs), BATCH_SIZE)):
    batch = all_docs[i:i+BATCH_SIZE]
    ids = [str(i+j) for j in range(len(batch))]
    vectorstore.add_documents(batch, ids=ids)

print("DONE")

100%|██████████| 666/666 [27:53<00:00,  2.51s/it]

DONE


In [23]:
print(vectorstore._collection.count())

66580


In [24]:
query = "what are the values of gitlab"

results = vectorstore.similarity_search(query, k=5)

for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(r.page_content[:500])


--- Result 1 ---

Title: GitLab Values
Description: Learn more about how we live our values at GitLab

Path: MyDrive > handbook > values > _index.md
Section: CREDIT > 👁️ Transparency {#transparency} > Public by default > Not public > Reproducibility

Content:
Enable everybody involved to come to the same conclusion as you. This not only involves reasoning, but also providing, for example: raw data and not just plots; scripts to automate tasks and not just the work they have done; and documenting steps while analy

--- Result 2 ---

Title: GitLab Values
Description: Learn more about how we live our values at GitLab

Path: MyDrive > handbook > values > _index.md
Section: CREDIT > 👁️ Transparency {#transparency} > Public by default > Not public > Say why, not just what

Content:
If you use generalized terms such as "industry standard" or "best practices," be sure to give context, as without context they can be seen as potentially vague or opaque.

Similarly, merely stating a single value

In [26]:
pip install rank_bm25

In [38]:
def process_file_bm25(path: str) -> List[Document]:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    fm, content = parse_frontmatter(raw)
    content = clean_text(content)

    title = fm.get("title") or os.path.basename(path)
    desc = fm.get("description", "")

    hierarchy = path_to_hierarchy(path)

    chunks = chunk_markdown(content)

    docs = []

    for headings, text in chunks:
        if not is_useful(text):
            continue

        section = " > ".join(headings) if headings else "General"

        sub_chunks = split_large(text)

        for sub in sub_chunks:
            formatted = f"""
{section}
{sub}
""".strip()

            docs.append(
                Document(
                    page_content=formatted.strip(),
                    metadata={
                        "source": path,
                        "section": section
                    }
                )
            )

    return docs

In [39]:
all_docs_bm25 = []

for path in tqdm(all_files, desc="Processing markdown files"):
    try:
        docs = process_file(path)
        all_docs_bm25.extend(docs)
    except Exception as e:
        print(path, e)

print("Total chunks:", len(all_docs_bm25))

Processing markdown files: 100%|██████████| 4114/4114 [00:19<00:00, 208.99it/s]

Total chunks: 66580


In [47]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(all_docs_bm25)
bm25_retriever.k = 6

In [48]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [49]:
from langchain_core.runnables import RunnableLambda

In [50]:
def hybrid_fn(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vector_retriever.invoke(query)

    seen = set()
    combined = []

    for doc in bm25_docs + vector_docs:
        key = doc.metadata["source"] + doc.metadata["section"]
        if key not in seen:
            seen.add(key)
            combined.append(doc)

    return combined

hybrid_retriever = RunnableLambda(hybrid_fn)

In [51]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("BAAI/bge-reranker-base")

def rerank_fn(inputs):
    query = inputs["query"]
    docs = inputs["docs"]

    pairs = [(query, d.page_content) for d in docs]
    scores = reranker_model.predict(pairs)

    ranked = [doc for _, doc in sorted(zip(scores, docs), reverse=True)]
    return ranked[:3]

reranker = RunnableLambda(rerank_fn)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
from langchain_core.runnables import RunnablePassthrough

In [53]:
retrieval_chain = (
    RunnablePassthrough()
    | {
        "query": lambda x: x,
        "docs": hybrid_retriever
    }
    | reranker
)

In [54]:
results = retrieval_chain.invoke("What are GitLab values?")

In [56]:
for i, r in enumerate(results):
    print(f"\n{'='*80}")
    print(f"RESULT {i+1}")
    print(f"{'='*80}")

    print(f"\nSource: {r.metadata.get('source')}")
    print(f"Section: {r.metadata.get('section')}")

    print("\nPreview:\n")
    print(r.page_content[:500])


RESULT 1

Source: /content/drive/MyDrive/handbook/values/_index.md
Section: CREDIT

Preview:

Title: GitLab Values
Description: Learn more about how we live our values at GitLab

Path: MyDrive > handbook > values > _index.md
Section: CREDIT

Content:
GitLab's six core values are
**🤝 Collaboration**,
**📈 Results for Customers**,
**⏱️ Efficiency**,
**🌐 Diversity, Inclusion & Belonging**,
**👣 Iteration**, and
**👁️ Transparency**,
and together they spell the **CREDIT** we give each other by assuming
good intent. We react to them with values emoji
and they are made actionable below.

RESULT 2

Source: /content/drive/MyDrive/handbook/values/_index.md
Section: CREDIT > What is not a value

Preview:

Title: GitLab Values
Description: Learn more about how we live our values at GitLab

Path: MyDrive > handbook > values > _index.md
Section: CREDIT > What is not a value

Content:
- All-remote isn't a value. It is something we do because it helps to practice our values of transparency, efficiency,

In [58]:
MULTI_QUERY_PROMPT = """
You are a retrieval expert.

Generate 3 alternative search queries for the given question.

Rules:
- Use different terminology that might appear in documentation
- Include related concepts or frameworks
- Be concise (no explanations)

Question:
{query}
"""

In [60]:
!zip -r /content/chroma_db.zip /content/drive/MyDrive/chroma_gitlab

  adding: content/drive/MyDrive/chroma_gitlab/ (stored 0%)
  adding: content/drive/MyDrive/chroma_gitlab/chroma.sqlite3 (deflated 61%)
  adding: content/drive/MyDrive/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/ (stored 0%)
  adding: content/drive/MyDrive/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/header.bin (deflated 54%)
  adding: content/drive/MyDrive/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/data_level0.bin


zip error: Interrupted (aborting)


In [62]:
!cp -r /content/drive/MyDrive/chroma_gitlab content/
!zip -r chroma_db.zip /content



zip error: Interrupted (aborting)


In [63]:
!cp -r /content/drive/MyDrive/chroma_gitlab /content/chroma_gitlab
!zip -r chroma_db.zip /content/chroma_gitlab

  adding: content/chroma_gitlab/ (stored 0%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/ (stored 0%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/header.bin (deflated 54%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/data_level0.bin (deflated 10%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/length.bin (deflated 76%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/link_lists.bin (deflated 79%)
  adding: content/chroma_gitlab/bc914786-b9d8-4b91-9f02-b9c94518ae51/index_metadata.pickle (deflated 60%)
  adding: content/chroma_gitlab/chroma.sqlite3 (deflated 61%)


In [64]:
!cp -r /content/drive/MyDrive/chroma_gitlab /content/chroma_gitlab_2
!zip -r chroma_db2.zip /content/chroma_gitlab_2

  adding: content/chroma_gitlab_2/ (stored 0%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/ (stored 0%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/header.bin (deflated 54%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/data_level0.bin (deflated 10%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/length.bin (deflated 76%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/link_lists.bin (deflated 79%)
  adding: content/chroma_gitlab_2/bc914786-b9d8-4b91-9f02-b9c94518ae51/index_metadata.pickle (deflated 60%)
  adding: content/chroma_gitlab_2/chroma.sqlite3 (deflated 61%)


In [66]:
from google.colab import files
files.download('chroma_db2.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [67]:
import pickle

with open("docs_bm25.pkl", "wb") as f:
    pickle.dump(all_docs_bm25, f)

In [68]:
files.download("docs_bm25.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>